In [ ]:
import requests
import json

shared_id = "dg_a64d1f7a-4d42-4638-9815-8188ecaadfd1"
AUTH_TOKEN = "XXX"

# Endpoint URL
url_dimensions = "https://datagroups.acc.appsvc.an.adobe.com/datagroups/1.0/users/sharedDimensions/" + shared_id
url_metrics = "https://datagroups.acc.appsvc.an.adobe.com/datagroups/1.0/users/sharedMetrics/" + shared_id

# Query parameters
params = {
    "locale": "en_US",
    "expansion": "tags,favorite,approved,storagePrecision,segmentable,dataSetIds,sourceFieldId,sourceFieldName,sourceFieldType,description,hideFromReporting,includeExcludeSetting,dataSetType,schemaPath,schemaType,type,required,tableName,baseTableName,allocation,duleLabels,deprecated,inheritedDataSetTypes,isArrayType,bucketingSetting,noValueOptionsSetting,booleanFormatSetting,defaultDimensionSort,persistenceSetting,multiValued,behaviorSetting,dateTimeFormatSetting,substringSetting,precisionSetting,summaryDataGroupingSetting"
}

# Headers
headers = {
    "accept": "application/json",
    "accept-language": "en-US,en;q=0.9,de-DE;q=0.8,de;q=0.7",
    "authorization": AUTH_TOKEN,
    "cache-control": "no-cache",
    "content-type": "application/json; charset=UTF-8",
    "pragma": "no-cache",
    "x-gw-ims-org-id": "4D6368F454EC41940A4C98A6@AdobeOrg",
    "x-gw-ims-user-id": "838642835DCD42960A495E68@375b28235c0505160a495c4f",
    "x-request-client-type": "UI",
    "x-request-id": "9566ba8526344f78a0ac9723291e634d"
}

In [ ]:
import pandas as pd
import requests
import json
import os

# --- 1. Your Helper Function (Provided by you) ---
def render_dimension(component_id, name, description, id, source, expiration=None):
    # --- FIX: Clean the source path immediately ---
    # If source is None, default to empty string to avoid errors
    clean_source = str(source) if source else ""
    
    # If it starts with "xdm.", remove the first 4 characters
    if clean_source.startswith("xdm."):
        clean_source = clean_source[4:]
    
    dimension = {
        "name": name,
        "description": description,
        "id": id,
        "sourceFieldId": clean_source  # Use the cleaned version here
    }
    
    # Handle Component ID (if we are updating an existing one)
    # Check for NaN (pandas empty value) or None
    if component_id and pd.notna(component_id):
        dimension["sharedComponentId"] = str(component_id)

    # Handle Expiration Logic
    if expiration in ["Visit", "Never", "Session", "Person Reporting Window"]:
        # Map your Excel values to CJA API values
        expiration_window = "sessions" # Default for Visit/Session
        
        if expiration in ["Never", "Person Reporting Window"]:
            expiration_window = "visitors"
            
        dimension["persistenceSetting"] = {
            "enabled": True,
            "allocationModel": {
                "func": "allocation-lastTouch_dim",
                "expiration": {
                    "func": "allocation-container",
                    "context": expiration_window
                }
            }
        }
    
    return dimension

# --- 2. API Wrapper (Modified to accept dynamic body) ---
def post_cja_dimensions(body, url, params, headers):
    try:
        # Note: Using PUT usually implies "Update/Replace". 
        # For creating new components, some Adobe APIs use POST. 
        # Keeping your PUT as per your snippet, assuming Bulk Upsert logic.
        response = requests.put(
            url + "/bulk",
            params=params,
            headers=headers,
            json=body
        )
        
        response.raise_for_status()
        
        return {
            "success": True,
            "data": response.json()
        }
        
    except requests.exceptions.HTTPError as e:
        return {
            "success": False,
            "error": str(e),
            "response_text": response.text
        }
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }

# --- 3. Main Logic ---
def sync_cja_dimensions(file_path, api_url, api_params, api_headers):
    sheet_name = "dimensions"
    
    # A. Load Excel
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    # Keep default_na=False to avoid empty strings becoming NaN floats
    df = pd.read_excel(file_path, sheet_name=sheet_name) 
    df['component_id'] = df['component_id'].astype('object')
    
    # --- THE FIX IS HERE ---
    # Replace NaN (float) with None (object), which becomes valid JSON 'null'
    df = df.where(pd.notnull(df), None)
    # -----------------------

    # B. Filter for items that need syncing
    # We look for "Pending Sync" or just "Pending" in status
    pending_mask = df['status'].astype(str).str.contains("Pending", case=False, na=False)
    pending_rows = df[pending_mask]

    if pending_rows.empty:
        print("No pending items found to sync.")
        return

    print(f"Found {len(pending_rows)} items to sync...")

    # C. Build Request Body
    request_body = []
    
    for index, row in pending_rows.iterrows():
        dim_payload = render_dimension(
            component_id=row['component_id'],
            name=row['name'],
            description=row['description'],
            id=row['id'],
            source=row['source'],
            expiration=row['expiration']
        )
        request_body.append(dim_payload)

    # D. Send to API
    # (Replace these with your actual credentials variables)
    print("Sending request to Adobe...")
    result = post_cja_dimensions(request_body, api_url, api_params, api_headers)

    if not result['success']:
        print("API Sync Failed!")
        print(f"Error: {result.get('error')}")
        print(f"Details: {result.get('response_text')}")
        return

    # E. Process Response & Update DataFrame
    print("Sync successful. Updating local file...")
    
    api_response_list = result['data']
    
    # Create a lookup dictionary from the API response for fast matching
    # Key = id (e.g., "eVar1"), Value = sharedComponentId (e.g., "6941...")
    # Note: API usually returns id as "variables/eVar1", so we strip "variables/"
    response_map = {}
    for item in api_response_list:
        clean_id = item.get('id', '').replace('variables/', '')
        comp_id = item.get('sharedComponentId')
        if clean_id and comp_id:
            response_map[clean_id] = comp_id

    # Iterate through the DataFrame and update rows that were just synced
    count_updated = 0
    for index, row in df.iterrows():
        # Only update if it was pending
        if "Pending" in str(row['status']):
            row_id = str(row['id'])
            
            # Check if this ID exists in our API response
            if row_id in response_map:
                df.at[index, 'component_id'] = response_map[row_id]
                df.at[index, 'status'] = "Synced"
                count_updated += 1
            else:
                print(f"Warning: {row_id} was sent but not found in response.")

    # F. Save back to Excel
    with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"Done! Updated {count_updated} rows in '{file_path}'.")

# --- usage setup ---
# You need to provide your actual API details here

filename = "cja-metrics-dimensions.xlsx"

# Run the sync
sync_cja_dimensions(filename, url_dimensions, params, headers)

Found 66 items to sync...
Sending request to Adobe...
Sync successful. Updating local file...
Done! Updated 66 rows in 'cja-metrics-dimensions.xlsx'.


In [13]:
import pandas as pd
import requests
import json
import os

# --- 1. Helper Function: Render Simple Metric Payload ---
def render_metric(component_id, name, description, id, source):
    # Clean the source path (remove 'xdm.' prefix if present)
    clean_source = str(source) if source else ""
    if clean_source.startswith("xdm."):
        clean_source = clean_source[4:]
        
    metric = {
        "name": name,
        "description": description,
        "id": id,
        "sourceFieldId": clean_source
        #"format": "integer",  # Required field, defaulting to integer
        #"tags": []
    }
    
    # If we are updating an existing component, attach the System ID
    if component_id and pd.notna(component_id):
        metric["sharedComponentId"] = str(component_id)

    return metric

# --- 2. Main Sync Logic ---
def sync_cja_metrics(file_path, api_url, api_params, api_headers):
    sheet_name = "metrics"
    
    # A. Load Excel
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    # Load and prep data
    df = pd.read_excel(file_path, sheet_name=sheet_name) 
    
    # Prevent "float" errors for empty IDs
    df['component_id'] = df['component_id'].astype('object')
    df = df.where(pd.notnull(df), None)

    # B. Filter for Pending items
    pending_mask = df['status'].astype(str).str.contains("Pending", case=False, na=False)
    pending_rows = df[pending_mask]

    if pending_rows.empty:
        print("No pending metrics found to sync.")
        return

    print(f"Found {len(pending_rows)} metrics to sync...")

    # C. Build Request Body
    request_body = []
    
    for index, row in pending_rows.iterrows():
        payload = render_metric(
            component_id=row['component_id'],
            name=row['name'],
            description=row['description'],
            id=row['id'],
            source=row['source']
        )
        request_body.append(payload)

    # D. Send to API
    print("Sending request to Adobe...")
    
    try:
        response = requests.put(
            api_url + "/bulk",
            params=api_params,
            headers=api_headers,
            json=request_body
        )
        response.raise_for_status()
        result = {"success": True, "data": response.json()}
        
    except Exception as e:
        print("API Sync Failed!")
        print(f"Error: {str(e)}")
        if 'response' in locals():
            print(f"Details: {response.text}")
        return

    # E. Process Response
    print("Sync successful. Updating local file...")
    
    api_response_list = result['data']
    
    # Map API response (metrics/event1 -> sharedComponentId)
    response_map = {}
    for item in api_response_list:
        # Metrics return IDs like "metrics/event1", we need just "event1" to match Excel
        clean_id = item.get('id', '').replace('metrics/', '') 
        comp_id = item.get('sharedComponentId')
        
        if clean_id and comp_id:
            response_map[clean_id] = comp_id

    # Update DataFrame
    count_updated = 0
    for index, row in df.iterrows():
        if "Pending" in str(row['status']):
            row_id = str(row['id'])
            
            # 1. Update Source in Excel (Clean xdm. prefix permanently)
            current_source = str(row['source']) if row['source'] else ""
            if current_source.startswith("xdm."):
                 df.at[index, 'source'] = current_source[4:]

            # 2. Update ID and Status
            if row_id in response_map:
                df.at[index, 'component_id'] = response_map[row_id]
                df.at[index, 'status'] = "Synced"
                count_updated += 1
            else:
                print(f"Warning: {row_id} sent but not returned by API.")

    # F. Save back to Excel (Safe Mode)
    try:
        # mode='a' appends/updates this sheet without deleting the 'dimensions' sheet
        with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            
        print(f"Done! Updated {count_updated} rows in '{sheet_name}'.")
        
    except PermissionError:
        print("\n!!! ERROR: Could not save the file. !!!")
        print(f"Please CLOSE '{file_path}' in Excel and run again.")
    except Exception as e:
        print(f"An unexpected error occurred while saving: {e}")

# --- Usage ---
sync_cja_metrics(filename, url_metrics, params, headers)

Found 295 metrics to sync...
Sending request to Adobe...
API Sync Failed!
Error: 404 Client Error: Not Found for url: https://datagroups.acc.appsvc.an.adobe.com/datagroups/1.0/users/sharedMetrics/dg_a64d1f7a-4d42-4638-9815-8188ecaadfd1/bulk/bulk?locale=en_US&expansion=tags%2Cfavorite%2Capproved%2CstoragePrecision%2Csegmentable%2CdataSetIds%2CsourceFieldId%2CsourceFieldName%2CsourceFieldType%2Cdescription%2ChideFromReporting%2CincludeExcludeSetting%2CdataSetType%2CschemaPath%2CschemaType%2Ctype%2Crequired%2CtableName%2CbaseTableName%2Callocation%2CduleLabels%2Cdeprecated%2CinheritedDataSetTypes%2CisArrayType%2CbucketingSetting%2CnoValueOptionsSetting%2CbooleanFormatSetting%2CdefaultDimensionSort%2CpersistenceSetting%2CmultiValued%2CbehaviorSetting%2CdateTimeFormatSetting%2CsubstringSetting%2CprecisionSetting%2CsummaryDataGroupingSetting
Details: {"errorCode":"resource_not_found","errorDescription":"Requested resource URL or path does not exist.  requestUrl=https://datagroups.acc.appsvc.